# Step 10 - Rewrite recommender, rule-based baseline (explicit bias only)

**Status: baseline. No human evaluation here (assigned to the research partner). Automatic checks only, and they use the same rules that detect the bias, so they show internal consistency, not quality.**

Goal (project objective 2): take an ad line flagged as biased and propose a neutral rewrite that changes **only** the biased line, leaving every other line byte-for-byte unchanged.

What this notebook does:
1. Loads the 484 units that Step 9 labelled `explicit` (gender, marital status, age) from the canonical Step 2 units.
2. Rewrites each one with deterministic rules (section 2).
3. Re-checks the rewritten text with the same rules and the saved Step 9 model (section 3).
4. Rewrites whole ads for the single-ad and multi-ad test (section 4).

What this notebook does **not** do: rewrite `implicit` units (668 in the corpus). Rule replacement is not safe for coded or appearance wording, and no seq2seq or LLM model has been trained or run, so those units are only listed for the next phase (section 5).

Inputs: `outputs/equal_step9_silver_baseline/step9_silver_labels.csv`, `step9_silver_model.joblib`. Outputs: `outputs/equal_step10_rewrite_baseline/`.

## 1. Environment and reproducibility log

In [1]:
import difflib, json, os, platform, re, sys
from pathlib import Path

import joblib, numpy as np, pandas as pd, sklearn

OUT = Path("outputs") / "equal_step10_rewrite_baseline"
OUT.mkdir(parents=True, exist_ok=True)
env = {"python": sys.version.split()[0], "platform": platform.platform(), "processor": platform.processor(),
       "logical_cpus": os.cpu_count(), "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
       "joblib": joblib.__version__, "gpu_used": False, "random_components": "none (deterministic rules)"}
try:
    import psutil
    env["ram_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
except ImportError:
    env["ram_gb"] = "not captured (psutil not installed)"
(OUT / "step10_environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
print(json.dumps(env, indent=2))

{
  "python": "3.12.9",
  "platform": "Windows-11-10.0.26200-SP0",
  "processor": "AMD64 Family 23 Model 113 Stepping 0, AuthenticAMD",
  "logical_cpus": 12,
  "numpy": "2.5.4",
  "pandas": "3.0.6",
  "scikit_learn": "1.9.1",
  "joblib": "1.6.0",
  "gpu_used": false,
  "random_components": "none (deterministic rules)",
  "ram_gb": 31.9
}


## 2. Rewrite rules

The detection regexes are copied from Step 6 and Step 9. The rewrite rules are new.

For each explicit unit, in this order:
0. **rule_false_positive**: the age pattern matched something that is not an age limit (for example "15-20 years of experience" or "up to 50 pounds"). The line is left unchanged and counted separately. This also shows a weakness of the Step 6 age rule.
1. **manual_review**: pregnancy, height or weight conditions, or a leftover fragment too short or too broken to stand alone (for example "LGBT" after removing "Male, Female"). No rewrite is proposed.
2. Remove the gender, marital-status and age phrases and the connectors left behind ("or", "and", "/", commas).
3. If **no content words remain**, the whole unit was only a restriction:
   - it contained a gender phrase: replace with the template `Open to all qualified candidates` (the wording from the project example);
   - it was only marital or age: delete the line (`deleted`).
4. If content remains, keep it (`partial_strip`), for example "With pleasing personality, female and single" becomes "With pleasing personality". Any remaining appearance wording is not touched; it is an implicit case.
5. **Template once per ad**: if an ad has several pure gender lines, the first becomes the template and the rest are deleted, so the neutral sentence is not repeated.

In [2]:
MALE_RE = re.compile(r"\b(?:male|males|men|gentlemen)\b", re.I)
FEMALE_RE = re.compile(r"\b(?:female|females|women|ladies)\b", re.I)
INCLUSIVE_RE = re.compile(r"\b(?:either\s+gender|any\s+gender|regardless\s+of\s+(?:sex|gender)|men\s+and\s+women|"
                          r"gender\s*[:\-]?\s*(?:any|open|not\s+required))\b", re.I)
MARITAL_RE = re.compile(r"\b(?:single|married|unmarried|widow(?:ed|er)?|marital|pregnan\w*)\b", re.I)
AGE_RE = re.compile(r"\b(?:\d{2}\s*(?:-|to)\s*\d{2}\s*(?:years|yrs|yo|y/o|years\s+old)|"
                    r"(?:below|under|not\s+(?:more|older)\s+than|up\s+to|maximum\s+(?:age\s+)?of)\s*\d{2}|"
                    r"age[sd]?\s*(?:range|limit|requirement)?\s*[:\-]?\s*\d{2})\b", re.I)
APPEAR_RE = re.compile(r"\b(?:good\s+looking|pleasing\s+personality|presentable|attractive|"
                       r"height|weight|physically\s+fit|heavy\s+lifting|tall|slim)\b", re.I)

TEMPLATE = "Open to all qualified candidates"
HOLD_RE = re.compile(r"pregnan\w*|\bheight\b|\bweight\b|centimeter|\d\s*'\s*\d|\bcm\b", re.I)
GENDER_PHRASE = re.compile(r"\b(?:preferably\s+)?(?:males?|females?|men|women)(?:\s+(?:only|applicants?))?\b", re.I)
MARITAL_PHRASE = re.compile(r"\b(?:preferably\s+)?(?:single(?:\s+mother)?|married|unmarried|widow(?:ed|er)?)\b", re.I)
AGE_PHRASE = re.compile(r"\b(?:not\s+more\s+than\s+\d{2}(?:\s*years?\s*old)?|\d{2}\s*(?:-|to)\s*\d{2}(?:\s*(?:years?\s*old|yrs|years))?|"
                        r"(?:below|under)\s*\d{2}(?:\s*years?\s*old)?)\b", re.I)
FILLER = {"or", "and", "a", "an", "the", "with", "of", "only", "preferably", "applicant", "applicants", "years", "old", "year", "age", "aged", "must", "be", "is", "are"}

def is_explicit_rule(s):
    return not INCLUSIVE_RE.search(s) and bool(MALE_RE.search(s) or FEMALE_RE.search(s) or MARITAL_RE.search(s) or AGE_RE.search(s))

def tidy(s):
    s = re.sub(r"\(\s*[,/&;:\-]*\s*\)", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*([,/&;:])\s*(?=[,/&;:]|$)", "", s)
    for _ in range(3):
        s = re.sub(r"(?:\s*[,/&]\s*|\s+)(?:and|or|with)\s*$", "", s, flags=re.I)
        s = re.sub(r"^\s*(?:and|or)\s+", "", s, flags=re.I)
    s = s.strip(" ,/&-;:")
    s = re.sub(r"^(?:preferably\s+)", "", s, flags=re.I)
    if s.count("(") < s.count(")"):
        s = s.rstrip(")").rstrip()
    if s.count("(") > s.count(")"):
        s = s.replace("(", "", 1).strip()
    return s[:1].upper() + s[1:]

NOT_AGE_RE = re.compile(r"\bexperience\b|\b(?:pounds|lbs|kg|kilos?)\b", re.I)
MODAL_START_RE = re.compile(r"^(?:can|must|should|will|may)\b", re.I)

def rewrite_unit(s):
    if HOLD_RE.search(s):
        return s, "manual_review"
    has_gender = bool(GENDER_PHRASE.search(s))
    if not has_gender and not MARITAL_PHRASE.search(s) and NOT_AGE_RE.search(s):
        return s, "rule_false_positive"
    rest = AGE_PHRASE.sub(" ", MARITAL_PHRASE.sub(" ", GENDER_PHRASE.sub(" ", s)))
    rest = tidy(rest)
    content = [w for w in re.findall(r"[A-Za-z]+", rest.lower()) if w not in FILLER]
    if not content:
        return (TEMPLATE, "template") if has_gender else ("", "deleted")
    if len(content) < 2 or MODAL_START_RE.match(rest):
        return s, "manual_review"
    return rest, "partial_strip"

for s in ["Male or Female", "Single/Married", "Not more than 40 years old", "With pleasing personality, female and single",
          "Preferably Male", "Female SRs must not be pregnant"]:
    print(f"{s!r:55} -> {rewrite_unit(s)}")

'Male or Female'                                        -> ('Open to all qualified candidates', 'template')
'Single/Married'                                        -> ('', 'deleted')
'Not more than 40 years old'                            -> ('', 'deleted')
'With pleasing personality, female and single'          -> ('With pleasing personality', 'partial_strip')
'Preferably Male'                                       -> ('Open to all qualified candidates', 'template')
'Female SRs must not be pregnant'                       -> ('Female SRs must not be pregnant', 'manual_review')


## 3. Apply to the corpus and check the result

In [3]:
lab = pd.read_csv("outputs/equal_step9_silver_baseline/step9_silver_labels.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
print("units:", len(lab), "| ads:", lab["record_id"].nunique(), "| silver classes:", lab["silver_bias_type"].value_counts().to_dict())
exp = lab[lab["silver_bias_type"] == "explicit"].copy()
print("explicit units:", len(exp), "| rule agrees on all of them:", bool(exp["sentence"].map(is_explicit_rule).all()))

res = exp["sentence"].map(rewrite_unit)
exp["rewrite"] = [r[0] for r in res]
exp["status"] = [r[1] for r in res]

# only the first pure-gender unit in an ad keeps the template; later ones are deleted
seen = set()
for i, r in exp.iterrows():
    if r["status"] == "template":
        if r["record_id"] in seen:
            exp.at[i, "rewrite"], exp.at[i, "status"] = "", "deleted"
        seen.add(r["record_id"])
display(exp["status"].value_counts().to_frame("units"))

units: 7971 | ads: 1500 | silver classes: {'none': 6819, 'implicit': 668, 'explicit': 484}
explicit units: 484 | rule agrees on all of them: True


,units
status,
template,410
deleted,59
manual_review,7
partial_strip,6
rule_false_positive,2


In [4]:
model = joblib.load("outputs/equal_step9_silver_baseline/step9_silver_model.joblib")
vec, clf = model["vectorizer"], model["classifier"]
kept = exp[exp["rewrite"] != ""].copy()
pred = clf.predict(vec.transform(kept["rewrite"]))
kept["model_after"] = pred
exp["rule_explicit_after"] = exp["rewrite"].map(lambda s: bool(s) and is_explicit_rule(s))
exp["appearance_after"] = exp["rewrite"].map(lambda s: bool(s) and bool(APPEAR_RE.search(s)))
exp = exp.join(kept["model_after"])
exp["edit_similarity"] = [round(difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio(), 3) for a, b in zip(exp["sentence"], exp["rewrite"])]

check = pd.DataFrame({
    "units": exp.groupby("status").size(),
    "rule_explicit_remaining": exp.groupby("status")["rule_explicit_after"].sum(),
    "model_still_explicit": exp.groupby("status")["model_after"].apply(lambda s: int((s == "explicit").sum())),
    "appearance_wording_remaining": exp.groupby("status")["appearance_after"].sum(),
}).fillna(0).astype(int)
display(check)
print("rewritten units that the Step 9 model still labels, by class:", kept["model_after"].value_counts().to_dict())
exp.to_csv(OUT / "step10_rewrite_units.csv", index=False, encoding="utf-8-sig")
mapping = exp.groupby(["sentence", "rewrite", "status"]).size().reset_index(name="occurrences").sort_values("occurrences", ascending=False)
mapping.to_csv(OUT / "step10_rewrite_mapping.csv", index=False, encoding="utf-8-sig")
print("distinct original texts:", len(mapping))
pd.set_option("display.max_colwidth", 90)
display(mapping.head(12))

,units,rule_explicit_remaining,model_still_explicit,appearance_wording_remaining
status,,,,
deleted,59,0,0,0
manual_review,7,7,7,3
partial_strip,6,0,0,4
rule_false_positive,2,2,2,0
template,410,0,0,0


rewritten units that the Step 9 model still labels, by class: {'none': 412, 'explicit': 9, 'implicit': 4}
distinct original texts: 50


,sentence,rewrite,status,occurrences
22,Male,Open to all qualified candidates,template,191
16,Female,Open to all qualified candidates,template,100
26,Male or Female,Open to all qualified candidates,template,57
32,Male/Female,Open to all qualified candidates,template,37
45,Single or Married,,deleted,14
44,Single,,deleted,13
47,Single/Married,,deleted,7
37,Not more than 40 years old,,deleted,5
49,"With pleasing personality, female and single",With pleasing personality,partial_strip,4
41,Preferably female,Open to all qualified candidates,template,4


**How to read section 3.** The `rule_explicit_remaining` column re-applies the same regexes that detected the bias, so a 0 only shows the rewrite removed what the rules look for. It does not show the rewrite reads well or is neutral to a person. The `model_still_explicit` column adds the Step 9 model as a second check; that model was trained on the same rule labels. Quality, neutrality and fluency are for the human evaluation.

### 3b. Content preservation at ad level

Because only flagged units are touched, every other unit is unchanged by construction. The table counts how many units and ads are affected.

In [5]:
n_units, n_ads = len(lab), lab["record_id"].nunique()
touched = exp[~exp["status"].isin(["manual_review", "rule_false_positive"])]
ad_tbl = lab.assign(is_explicit=lab["silver_bias_type"] == "explicit").groupby("record_id").agg(units=("sentence_id", "size"), explicit_units=("is_explicit", "sum"))
cleared = exp.groupby("record_id").apply(lambda g: not ((g["status"] == "manual_review") | g["rule_explicit_after"]).any(), include_groups=False)
summary = pd.Series({
    "ads": n_ads, "units": n_units,
    "ads_with_explicit_units": int((ad_tbl["explicit_units"] > 0).sum()),
    "explicit_units": len(exp),
    "units_changed_or_removed": len(touched),
    "units_unchanged_by_construction": n_units - len(touched),
    "share_of_all_units_changed": round(len(touched) / n_units, 4),
    "ads_fully_cleared_of_explicit_by_rules": int(cleared.sum()),
    "ads_needing_manual_review": int(exp[exp["status"] == "manual_review"]["record_id"].nunique()),
    "rule_false_positive_units": int((exp["status"] == "rule_false_positive").sum()),
    "mean_edit_similarity_partial_strip": round(exp.loc[exp["status"] == "partial_strip", "edit_similarity"].mean(), 3) if (exp["status"] == "partial_strip").any() else None,
})
summary.to_csv(OUT / "step10_corpus_summary.csv", header=["value"])
display(summary.to_frame("value"))
display(exp[exp["status"].isin(["partial_strip", "manual_review", "rule_false_positive"])].drop_duplicates("sentence")[["status", "sentence", "rewrite"]].reset_index(drop=True))

,value
ads,1500.0000
units,7971.0000
ads_with_explicit_units,429.0000
explicit_units,484.0000
units_changed_or_removed,475.0000
units_unchanged_by_construction,7496.0000
share_of_all_units_changed,0.0596
ads_fully_cleared_of_explicit_by_rules,420.0000
ads_needing_manual_review,7.0000
rule_false_positive_units,2.0000


,status,sentence,rewrite
0,manual_review,Female SRs must not be pregnant,Female SRs must not be pregnant
1,manual_review,Must be atleast one meter and sixty-two centimeters (1.62 cm) in height for Male and o...,Must be atleast one meter and sixty-two centimeters (1.62 cm) in height for Male and o...
2,manual_review,"18-25 years old, male at least 5'5"" can work under pressure","18-25 years old, male at least 5'5"" can work under pressure"
3,partial_strip,HR Recruitment Staff (Preferably Male),HR Recruitment Staff
4,manual_review,"At least 5'4"" height for Male and 5'0"" for Female","At least 5'4"" height for Male and 5'0"" for Female"
5,partial_strip,"With pleasing personality, female and single",With pleasing personality
6,rule_false_positive,"Ability to lift, carry, push, and pull up to 50 pounds","Ability to lift, carry, push, and pull up to 50 pounds"
7,manual_review,At least 5’4”(Female) in height,At least 5’4”(Female) in height
8,partial_strip,"Male, with valid driver’s license (4-wheel and motorcycle)",With valid driver’s license (4-wheel and motorcycle)
9,manual_review,"Male, Female, LGBT","Male, Female, LGBT"


## 4. Rewrite test: single ad and multiple ads

`rewrite_ad(text)` keeps every line and its bullet (`~`, `-`, and so on) exactly as written and replaces only lines the rules flag as explicit. Lines marked `manual_review` are left unchanged and reported. Contact lines are not masked here because the output is meant to be a usable ad.

In [6]:
BULLET_RE = re.compile(r"^(\s*(?:[~\u2022\u25cf\u25aa\u25e6\u2023\u25cb\u25a0\u25a1\u25c6\u27a2\u27a4\u2713\u2714]+|[-*](?=\s|$))?\s*)(.*)$")

def rewrite_ad(text):
    out, log, template_used = [], [], False
    for line in str(text).splitlines():
        m = BULLET_RE.match(line)
        prefix, body = m.group(1), m.group(2)
        if body and is_explicit_rule(body):
            new, status = rewrite_unit(body)
            if status == "template" and template_used:
                new, status = "", "deleted"
            template_used = template_used or status == "template"
            log.append({"original": body, "rewrite": new, "status": status})
            if status in ("manual_review", "rule_false_positive"):
                out.append(line)
            elif new:
                out.append(prefix + new)
        else:
            out.append(line)
    return "\n".join(out), pd.DataFrame(log, columns=["original", "rewrite", "status"])

def rewrite_ads(ads):
    if not isinstance(ads, dict):
        ads = {f"ad_{i + 1}": a for i, a in enumerate(ads)}
    rows, texts = [], {}
    for k, t in ads.items():
        new, lg = rewrite_ad(t)
        texts[k] = new
        for r in lg.to_dict("records"):
            rows.append({"ad_id": k, **r})
        if lg.empty:
            rows.append({"ad_id": k, "original": "", "rewrite": "", "status": "no explicit line"})
    return texts, pd.DataFrame(rows)

In [7]:
PUREGOLD = """Puregold

Human Resource Assistant

Qualifications:
~Female/Male
~Graduate of BS Psychology, BSBA Human Resource Management, BS Behavioral Science or other related courses
~With at least 1 year experience in human resources functions
~With knowledge in implementing all human resources faucet

For inquiries and referral, visit or contact:
~web.facebook.com/LMDPESOMarinduqueOfficial
~lmdpeso@yahoo.com
~Mobile No.: 0912-673-0915"""

new_text, log = rewrite_ad(PUREGOLD)
print(new_text); print()
display(log)
orig_lines, new_lines = PUREGOLD.splitlines(), new_text.splitlines()
print("lines before/after:", len(orig_lines), len(new_lines), "| lines identical:", sum(a == b for a, b in zip(orig_lines, new_lines)))

Puregold

Human Resource Assistant

Qualifications:
~Open to all qualified candidates
~Graduate of BS Psychology, BSBA Human Resource Management, BS Behavioral Science or other related courses
~With at least 1 year experience in human resources functions
~With knowledge in implementing all human resources faucet

For inquiries and referral, visit or contact:
~web.facebook.com/LMDPESOMarinduqueOfficial
~lmdpeso@yahoo.com
~Mobile No.: 0912-673-0915



,original,rewrite,status
0,Female/Male,Open to all qualified candidates,template


lines before/after: 14 14 | lines identical: 13


In [8]:
TEST_ADS = {
    "puregold_hr_assistant": PUREGOLD,
    "yamaha": "Yamaha\n\nProduction Associates (70)\n\nQualifications:\n~At least High School Graduate or 2-year Vocational Graduate\n~With or without work experience\n~Must be willing to render overtime",
    "synthetic_test": "Sales Associate\n\nQualifications:\n~Single, 18-25 years old\n~With pleasing personality, female and single\n~Dependable and willing to be trained\n~Preferably Male",
}
texts, multi_log = rewrite_ads(TEST_ADS)
multi_log.to_csv(OUT / "step10_test_rewrite_log.csv", index=False, encoding="utf-8-sig")
display(multi_log)
print(texts["synthetic_test"])

,ad_id,original,rewrite,status
0,puregold_hr_assistant,Female/Male,Open to all qualified candidates,template
1,yamaha,,,no explicit line
2,synthetic_test,"Single, 18-25 years old",,deleted
3,synthetic_test,"With pleasing personality, female and single",With pleasing personality,partial_strip
4,synthetic_test,Preferably Male,Open to all qualified candidates,template


Sales Associate

Qualifications:
~With pleasing personality
~Dependable and willing to be trained
~Open to all qualified candidates


## 5. What is not rewritten

Implicit units stay unchanged and are only counted. Candidate approaches for them (retrieval of neutral candidates, constrained seq2seq or LLM rewriting) are described in `literature-support-for-methods.md`. None has been run, so there are no results for them.

In [9]:
imp = lab[lab["silver_bias_type"] == "implicit"]
print("implicit units not rewritten:", len(imp), "| distinct texts:", imp["sentence"].nunique())
display(imp["sentence"].value_counts().head(10).to_frame("occurrences"))

implicit units not rewritten: 668 | distinct texts: 446


,occurrences
sentence,
With pleasing personality,32
Physically fit,10
"Strong leadership, analytical and interpersonal skills",9
Good communication and interpersonal skills,7
"Must be honest, hardworking and can work independently in a time sensitive environment",7
Good interpersonal skills,6
Has good positive attitude and trustworthy,6
With active PRC License,5
Honest and Trustworthy (With Integrity),5


## 6. Limitations

- Deterministic rules cover only gender, marital-status and age wording; they do not handle implicit bias.
- Residual checks use the same rules that detect the bias, so they cannot find bias the rules miss.
- The template wording comes from the project example; other wording has not been compared.
- No fluency, readability, neutrality or content-preservation judgement by people is made here. That evaluation belongs to the research partner.
- The units are the masked Step 2 units; employer and contact blocks are held separately, so a full ad has to be rebuilt from Step 1 before publication.